# 5. Balanced assignment problem

Lecture: *Week 6 - Integer Linear Programming - Further Problems*, PDF pages 16–23. The first of three versions
of the assignment model.

**The problem.** Four workers must do four jobs, one job each. The table gives the time each worker needs for
each job (page 20):

| | Job 1 | Job 2 | Job 3 | Job 4 |
| --- | --- | --- | --- | --- |
| Worker 1 | 9 | 2 | 7 | 8 |
| Worker 2 | 6 | 4 | 3 | 7 |
| Worker 3 | 5 | 8 | 1 | 8 |
| Worker 4 | 7 | 6 | 9 | 4 |

Minimise the total time. $x_{uv} = 1$ if worker $u$ does job $v$ (page 22):

minimise $\sum_{u \in W} \sum_{v \in J} c_{uv} x_{uv}$ subject to $\sum_{v \in J} x_{uv} = 1$ for every worker $u$,
$\sum_{u \in W} x_{uv} = 1$ for every job $v$, $x_{uv} \in \{0, 1\}$.

"Pick one cell in every row and every column": a perfect matching in the bipartite graph of workers and jobs.
There are $4! = 24$ possible assignments.

In the code workers and jobs are numbered from 0: `x[i, j]` is worker $i+1$ doing job $j+1$, and `w[i][j]` is
its time.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** Total time **13**: worker 1 → job 2, worker 2 → job 1, worker 3 → job 3, worker 4 → job 4
(`x[0,1]`, `x[1,0]`, `x[2,2]`, `x[3,3]` = 1), $2 + 6 + 1 + 4 = 13$, as on page 23. It is the only optimal
assignment.

*Original script: `5 Source Files/Python/balanced_assignment.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools.

In [ ]:
from pyomo.environ import *

Create the model, a 4 × 4 grid of binary variables and the table of times.

In [ ]:
# Create the model and the binary variables: x[i, j] = 1 if worker i+1 does job j+1
model = ConcreteModel()
model.x = Var(range(4), range(4), within=Binary)

w = [   # w[i][j] = time for worker i+1 to do job j+1 (rows = workers, columns = jobs)
    [9, 2, 7, 8],
    [6, 4, 3, 7],
    [5, 8, 1, 8],
    [7, 6, 9, 4]
]

The objective: the total time of the chosen (worker, job) pairs, minimised.

In [ ]:
# Create the optimization problem
model.obj = Objective(expr=sum(sum(w[i][j] * model.x[i, j] for j in range(4)) for i in range(4)), sense=minimize)

Two groups of constraints: every worker does exactly one job (one per row of the table), and every job gets exactly one worker (one per column).

In [ ]:
# Define the equality constraints for workers and jobs
model.worker_constraints = ConstraintList()
model.job_constraints = ConstraintList()

for i in range(4):
    model.worker_constraints.add(sum(model.x[i, j] for j in range(4)) == 1)   # worker i+1 does exactly one job
    
for j in range(4):
    model.job_constraints.add(sum(model.x[i, j] for i in range(4)) == 1)   # job j+1 gets exactly one worker

HiGHS is the solver; unlike GLPK it runs in the browser.

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

`pprint()` prints the model: 16 variables, 4 worker constraints and 4 job constraints.

In [ ]:
# Display the model
model.pprint()

Solve. In the browser HiGHS prints its log here even with `tee=False` (on a desktop it stays quiet); the *Solving report* at the end should say *Status Optimal*.

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

`display()` prints all 16 variables; the four with value 1 are the assignment, and the objective is the total time.

In [ ]:
# Display the results
model.display()

## Try it

Change `within=Binary` to `within=NonNegativeReals`, so that the solver may also choose fractions such as
$x_{uv} = 0.5$, and run all cells again. Does the answer change?

<details><summary>Answer</summary>

No: the total is still 13 and every variable is still 0 or 1. The assignment problem is special: its LP
relaxation always has a whole-number optimal corner point, because of the structure of the constraints
(each variable appears in exactly one row constraint and one column constraint). Most integer programs, such
as the fire stations, do not behave like this. Change it back to `Binary` afterwards.
</details>